# Preparação dos Dados — Bank Marketing

## Objetivo

Este notebook realiza a preparação da base Bank Marketing utilizada no Datathon da Pós-Tech em Machine Learning Engineering da FIAP.

As decisões desta etapa são baseadas nas observações realizadas durante a Análise Exploratória de Dados (EDA).

Os principais tratamentos realizados são:

- remoção de registros duplicados;
- remoção da variável `duration` devido ao vazamento de informação;
- transformação da variável alvo em recompensa binária;
- definição do canal de contato como braço da política adaptativa;
- tratamento da variável `pdays`;
- criação de variáveis de contexto para utilização na estratégia adaptativa;
- preservação dos valores `unknown` como uma categoria explícita;
- geração da base processada para as próximas etapas do projeto.

In [54]:
from pathlib import Path

import numpy as np
import pandas as pd

In [55]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bank-additional-full.csv"

df = pd.read_csv(RAW_DATA_PATH, sep=";")

df.shape

(41188, 21)

In [56]:
processed_df = df.copy()

In [57]:
processed_df.insert(
    0,
    "interaction_id",
    np.arange(len(processed_df))
)

In [58]:
initial_rows = len(processed_df)

processed_df = (
    processed_df
    .drop_duplicates(
        subset=[column for column in processed_df.columns if column != "interaction_id"]
    )
    .reset_index(drop=True)
)

removed_duplicates = initial_rows - len(processed_df)

print(f"Registros antes: {initial_rows}")
print(f"Duplicados removidos: {removed_duplicates}")
print(f"Registros depois: {len(processed_df)}")

Registros antes: 41188
Duplicados removidos: 12
Registros depois: 41176


### Registros duplicados

Os 12 registros completamente duplicados identificados durante a análise exploratória foram removidos.

O identificador `interaction_id` preserva a posição original das observações para permitir análises sequenciais nas próximas etapas.

In [59]:
processed_df = processed_df.drop(columns=["duration"])

In [60]:
"duration" in processed_df.columns

False

### Remoção da variável `duration`

A variável `duration` foi removida por apresentar vazamento de informação.

A duração total da ligação somente é conhecida após o contato com o cliente. Portanto, essa informação não estaria disponível no momento em que a política precisasse escolher uma ação.

Sua utilização produziria uma avaliação artificialmente otimista da solução.

In [61]:
processed_df["reward"] = processed_df["y"].map({
    "no": 0,
    "yes": 1
})

In [62]:
processed_df[["y", "reward"]].head()

,y,reward
0,no,0
1,no,0
2,no,0
3,no,0
4,no,0


In [63]:
processed_df["reward"].value_counts()

reward
0    36537
1     4639
Name: count, dtype: int64

In [64]:
processed_df = processed_df.drop(columns=["y"])

### Definição da recompensa

A variável alvo original `y` foi convertida para uma recompensa binária:

- `1`: o cliente realizou a assinatura do depósito a prazo;
- `0`: o cliente não realizou a assinatura.

Essa representação será utilizada pelo algoritmo adaptativo nas próximas etapas.

In [65]:
processed_df["arm"] = processed_df["contact"]

In [66]:
processed_df["arm"].value_counts()

arm
cellular     26135
telephone    15041
Name: count, dtype: int64

In [67]:
processed_df = processed_df.drop(columns=["contact"])

### Definição dos braços da política adaptativa

A variável `contact` foi utilizada para representar os braços da estratégia adaptativa.

Os dois braços disponíveis são:

- `cellular`;
- `telephone`.

Dessa forma, a política adaptativa terá como objetivo aprender qual canal de contato apresenta maior recompensa esperada de acordo com o contexto disponível.

A coluna original `contact` foi removida das features para evitar que a ação escolhida seja utilizada como informação de entrada da própria decisão.

In [68]:
processed_df["previously_contacted"] = (
    processed_df["pdays"] != 999
).astype(int)

In [69]:
processed_df["pdays_since_previous"] = (
    processed_df["pdays"]
    .where(processed_df["pdays"] != 999)
    .astype("Int64")
)

In [70]:
processed_df[
    ["pdays", "previously_contacted", "pdays_since_previous"]
].head(10)

,pdays,previously_contacted,pdays_since_previous
0,999,0,<NA>
1,999,0,<NA>
2,999,0,<NA>
3,999,0,<NA>
4,999,0,<NA>
5,999,0,<NA>
6,999,0,<NA>
7,999,0,<NA>
8,999,0,<NA>
9,999,0,<NA>


In [71]:
print(processed_df["previously_contacted"].value_counts())

processed_df.loc[
    processed_df["previously_contacted"] == 1,
    ["pdays", "previously_contacted", "pdays_since_previous"]
].head(10)

previously_contacted
0    39661
1     1515
Name: count, dtype: int64


,pdays,previously_contacted,pdays_since_previous
24101,6,1,6
24257,4,1,4
24272,4,1,4
24390,3,1,3
24475,4,1,4
24606,5,1,5
24792,5,1,5
24843,1,1,1
24903,6,1,6
25046,4,1,4


In [72]:
processed_df = processed_df.drop(columns=["pdays"])

### Tratamento da variável `pdays`

Na base original, o valor `999` em `pdays` representa clientes que não haviam sido contatados anteriormente, e não um intervalo literal de 999 dias.

Para representar corretamente essa informação foram criadas duas variáveis:

- `previously_contacted`: indica se houve contato anterior;
- `pdays_since_previous`: quantidade de dias desde o contato anterior, quando existente.

A variável original `pdays` foi removida.

In [73]:
unknown_counts = {}

for column in processed_df.select_dtypes(include="object").columns:
    count = (processed_df[column] == "unknown").sum()

    if count > 0:
        unknown_counts[column] = count

pd.Series(unknown_counts).sort_values(ascending=False)

default      8596
education    1730
housing       990
loan          990
job           330
marital        80
dtype: int64

### Tratamento de valores `unknown`

Os valores `unknown` foram mantidos como categorias explícitas.

Não foi realizada imputação dessas informações, pois substituir categorias desconhecidas pela moda ou por outra categoria adicionaria uma suposição que não é sustentada pela base.

A política poderá, portanto, tratar clientes com informações desconhecidas como parte válida do contexto disponível.

In [74]:
processed_df["age_group"] = pd.cut(
    processed_df["age"],
    bins=[-np.inf, 29, 39, 49, 59, np.inf],
    labels=[
        "under_30",
        "30_39",
        "40_49",
        "50_59",
        "60_plus"
    ]
)

In [75]:
processed_df["age_group"].value_counts().sort_index()

age_group
under_30     5667
30_39       16933
40_49       10523
50_59        6861
60_plus      1192
Name: count, dtype: int64

### Contexto da política adaptativa

Foi criada a variável `age_group` para permitir a utilização da idade de forma segmentada na política adaptativa.

Inicialmente, a estratégia utilizará um contexto simplificado composto por `age_group` e `poutcome`.

Essa escolha busca incorporar características do cliente e seu histórico de campanhas sem criar uma quantidade excessiva de segmentos com poucos registros.

A definição poderá ser revisada durante os experimentos da próxima etapa.

In [76]:
processed_df.head()

,interaction_id,age,job,marital,education,default,housing,loan,month,day_of_week,...,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,reward,arm,previously_contacted,pdays_since_previous,age_group
0,0,56,housemaid,married,basic.4y,no,no,no,may,mon,...,1.1,93.994,-36.4,4.857,5191.0,0,telephone,0,<NA>,50_59
1,1,57,services,married,high.school,unknown,no,no,may,mon,...,1.1,93.994,-36.4,4.857,5191.0,0,telephone,0,<NA>,50_59
2,2,37,services,married,high.school,no,yes,no,may,mon,...,1.1,93.994,-36.4,4.857,5191.0,0,telephone,0,<NA>,30_39
3,3,40,admin.,married,basic.6y,no,no,no,may,mon,...,1.1,93.994,-36.4,4.857,5191.0,0,telephone,0,<NA>,40_49
4,4,56,services,married,high.school,no,no,yes,may,mon,...,1.1,93.994,-36.4,4.857,5191.0,0,telephone,0,<NA>,50_59


In [77]:
processed_df.shape

(41176, 23)

In [78]:
processed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41176 entries, 0 to 41175
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   interaction_id        41176 non-null  int64   
 1   age                   41176 non-null  int64   
 2   job                   41176 non-null  object  
 3   marital               41176 non-null  object  
 4   education             41176 non-null  object  
 5   default               41176 non-null  object  
 6   housing               41176 non-null  object  
 7   loan                  41176 non-null  object  
 8   month                 41176 non-null  object  
 9   day_of_week           41176 non-null  object  
 10  campaign              41176 non-null  int64   
 11  previous              41176 non-null  int64   
 12  poutcome              41176 non-null  object  
 13  emp.var.rate          41176 non-null  float64 
 14  cons.price.idx        41176 non-null  float64 
 15  co

In [79]:
print(f"Registros processados: {len(processed_df)}")
print(f"IDs únicos: {processed_df['interaction_id'].nunique()}")

print("\nValidação das colunas removidas:")
print("duration presente:", "duration" in processed_df.columns)
print("y presente:", "y" in processed_df.columns)
print("contact presente:", "contact" in processed_df.columns)

print("\nRewards:")
print(processed_df["reward"].value_counts())

print("\nArms:")
print(processed_df["arm"].value_counts())

print("\nClientes contatados anteriormente:")
print(processed_df["previously_contacted"].value_counts())

print("\nAge groups:")
print(processed_df["age_group"].value_counts())

Registros processados: 41176
IDs únicos: 41176

Validação das colunas removidas:
duration presente: False
y presente: False
contact presente: False

Rewards:
reward
0    36537
1     4639
Name: count, dtype: int64

Arms:
arm
cellular     26135
telephone    15041
Name: count, dtype: int64

Clientes contatados anteriormente:
previously_contacted
0    39661
1     1515
Name: count, dtype: int64

Age groups:
age_group
30_39       16933
40_49       10523
50_59        6861
under_30     5667
60_plus      1192
Name: count, dtype: int64


In [80]:
assert processed_df["interaction_id"].is_unique

assert set(processed_df["reward"].unique()) == {0, 1}

assert set(processed_df["arm"].unique()) == {
    "cellular",
    "telephone"
}

assert processed_df.loc[
    processed_df["previously_contacted"] == 0,
    "pdays_since_previous"
].isna().all()

assert processed_df.loc[
    processed_df["previously_contacted"] == 1,
    "pdays_since_previous"
].notna().all()

assert "duration" not in processed_df.columns
assert "y" not in processed_df.columns
assert "contact" not in processed_df.columns

print("Todas as validações foram executadas com sucesso.")

Todas as validações foram executadas com sucesso.


In [81]:
context_arm_distribution = pd.crosstab(
    [
        processed_df["age_group"],
        processed_df["poutcome"]
    ],
    processed_df["arm"]
)

context_arm_distribution

arm                    cellular  telephone
age_group poutcome                        
under_30  failure           626         48
          nonexistent      2919       1761
          success           286         27
30_39     failure          1659        141
          nonexistent      8760       5923
          success           415         35
40_49     failure           890         52
          nonexistent      5132       4242
          success           195         12
50_59     failure           551         43
          nonexistent      3542       2526
          success           183         16
60_plus   failure           226         16
          nonexistent       560        186
          success           191         13

In [82]:
print(
    "Quantidade de contextos:",
    context_arm_distribution.shape[0]
)

print(
    "Existe contexto sem algum braço:",
    (context_arm_distribution == 0).any().any()
)

Quantidade de contextos: 15
Existe contexto sem algum braço: False


### Definição final para a política adaptativa

Para os experimentos da próxima etapa, foram definidos:

- **Contexto:** combinação entre `age_group` e `poutcome`;
- **Braços:** `cellular` e `telephone`;
- **Recompensa:** variável binária `reward`, em que 1 representa conversão e 0 representa ausência de conversão.

Foram identificados 15 contextos possíveis na base processada. Todos possuem registros para ambos os braços, permitindo sua utilização nos experimentos com a política adaptativa.

As demais variáveis foram preservadas na base processada para manter as informações disponíveis para análises e possíveis extensões futuras da solução.

In [83]:
PROCESSED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bank_marketing_processed.csv"
)

PROCESSED_DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

processed_df.to_csv(
    PROCESSED_DATA_PATH,
    index=False
)

print(f"Dataset salvo em: {PROCESSED_DATA_PATH}")

Dataset salvo em: c:\codes\fiap-datathon-mlet\data\processed\bank_marketing_processed.csv


In [84]:
validation_df = pd.read_csv(PROCESSED_DATA_PATH)

validation_df.shape

(41176, 23)

# Conclusão da Preparação dos Dados

A base Bank Marketing foi preparada para os experimentos com a política adaptativa.

Durante esta etapa:

- foram removidos 12 registros completamente duplicados;
- a ordem original das interações foi preservada por meio de `interaction_id`;
- a variável `duration` foi removida para evitar vazamento de informação;
- a variável alvo foi convertida para uma recompensa binária;
- o canal de contato foi definido como o braço da política adaptativa;
- a variável `pdays` foi transformada para representar adequadamente a existência e o tempo desde contatos anteriores;
- valores `unknown` foram preservados como categorias explícitas;
- foi criada a variável `age_group`;
- foi definido um contexto simplificado baseado em `age_group` e `poutcome`;
- a base processada foi salva para utilização nos experimentos posteriores.

A próxima etapa consiste na implementação e comparação entre uma política baseline e uma estratégia adaptativa baseada em Thompson Sampling.